# NIDAR Survivor Detection — Colab Training (YOLO26n, GPU)
Baseline: `yolo26n.pt` (Jan 2026 SOTA, small-target ProgLoss+STAL) with `yolo11n.pt` fallback. All paths configurable via `configs/training.yaml`.

In [ ]:
import torch, sys
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a Colab GPU: Runtime > Change runtime type > T4'

In [ ]:
!pip -q install ultralytics>=8.3 pyyaml tqdm matplotlib
from ultralytics import YOLO; import ultralytics; print('ultralytics', ultralytics.__version__)

In [ ]:
# Attach data: upload datasets/processed/ (with data.yaml) or mount Drive and symlink it.
from pathlib import Path
assert Path('datasets/processed/data.yaml').exists(), 'Upload datasets/processed/ first (see docs/DATASETS.md)'
print(open('datasets/processed/data.yaml').read())

In [ ]:
import yaml, random, numpy as np
cfg = yaml.safe_load(open('configs/training.yaml'))
seed = cfg['train']['seed']
random.seed(seed); np.random.seed(seed); import torch; torch.manual_seed(seed)
print(cfg)

In [ ]:
from ultralytics import YOLO
pref, fb = cfg['model']['weights'], cfg['model']['fallback_weights']
try:
    model = YOLO(pref); used = pref
except Exception as e:
    print(f'{pref} failed ({e}); falling back to {fb}'); model = YOLO(fb); used = fb
print('weights:', used)

In [ ]:
t, a = cfg['train'], cfg['augment']
results = model.train(data=cfg['data'], epochs=t['epochs'], imgsz=t['imgsz'], batch=t['batch'],
    patience=t['patience'], project=t['project'], name=t['name'], seed=t['seed'],
    hsv_h=a['hsv_h'], hsv_s=a['hsv_s'], hsv_v=a['hsv_v'], degrees=a['degrees'],
    translate=a['translate'], scale=a['scale'], shear=a['shear'], fliplr=a['fliplr'],
    flipud=a['flipud'], mosaic=a['mosaic'], mixup=a['mixup'], copy_paste=a['copy_paste'])
print(results)

In [ ]:
# Validate + plot
m = model.val(split='test')
print(f"mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}")
from pathlib import Path
run = Path(t['project'])/t['name']
print('best:', run/'weights'/'best.pt')
try:
    from IPython.display import Image
    display(Image(str(run/'results.png')))
    display(Image(str(run/'confusion_matrix.png')))
except Exception as e:
    print('plots not available:', e)

In [ ]:
# Export NCNN for the Pi (copy the model dir to the aircraft)
model = YOLO(str(Path(t['project'])/t['name']/'weights'/'best.pt'))
print(model.export(format='ncnn', imgsz=640))